#Traitement du tableau binaire de segmentation des halos



*   Le fichier excel des intéractions généré à partir de la segmentation de SAM2 contient 39 sheets, un sheet par souche, chaque sheet comprend les intéractions de celle-ci avec les autres
*   L'idée est de faire un traitement pour le convertir en tableau binaire prêt à l'analyse



In [1]:
# Packages nécessaires
import re
import pandas as pd

Le code qui suit lit le fichier Excel qui vient de du script de segmentation avec SAM2 qui contient plusieurs feuilles (onglets).
Chaque feuille correspond à une souche “cible”, et à l’intérieur de chaque feuille se trouvent des observations sur différentes souches “productrices” testées contre elle.

Le code produit un fichier CSV (halos_couples.csv) qui liste, pour chaque couple “productrice-cible”, la valeur de “halo” correspondant.

In [2]:
INPUT_XLSX = "../data/sortie/halos_par_souche.xlsx"   # Fichier Excel d’entrée
SHEETS_INCLUDE = None                  # None = toutes les feuilles; sinon: ['1009','1012',...]

# GRILLE : index_spot -> SOUCHE PRODUCTRICE
MAPPING = {
    "0_0":"982","0_1":"959","0_2":"958","0_3":"955","0_4":"952",
    "1_0":"988","1_1":"987","1_2":"986","1_3":"985","1_4":"984",
    "2_0":"995","2_1":"994","2_2":"993","2_3":"992","2_4":"990",
    "3_0":"1003","3_1":"1001","3_2":"999","3_3":"998","3_4":"996",
    "4_0":"1012","4_1":"1011","4_2":"1009","4_3":"1008","4_4":"1006",
    "5_0":"1020","5_1":"1019","5_2":"1017","5_3":"1015","5_4":"1014",
    "6_0":"1036","6_1":"1033","6_2":"1032","6_3":"1028","6_4":"1025",
    "7_0":"M63","7_1":"1054","7_2":"1048","7_3":"1044","7_4":"1041"
}

# FONCTIONS UTILITAIRES
def normalize_cible_name(sheet_name: str) -> str:
    """Extrait un nom de souche 'propre' à partir du nom d'onglet."""
    m = re.match(r"\s*([A-Za-z0-9]+)", str(sheet_name))
    return m.group(1) if m else str(sheet_name).strip()

def halo_to_10_or_blank(v):
    """Convertit la valeur 'halo' en 1, 0 ou vide."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, bool):
        return 1 if v else 0
    s = str(v).strip().lower()
    if s in {"vrai", "true", "1", "oui", "o", "x"}:
        return 1
    if s in {"faux", "false", "0", "non"}:
        return 0
    if s in {"na", "nan", ""}:
        return ""
    try:
        f = float(s.replace(",", "."))
        if f == 1.0:
            return 1
        if f == 0.0:
            return 0
    except:
        pass
    return ""

# LECTURE ET COMPTAGE DES FEUILLES
sheets = pd.read_excel(INPUT_XLSX, sheet_name=None)
sheet_names = SHEETS_INCLUDE if SHEETS_INCLUDE is not None else sheets.keys()

counts = {}
for cible, df in sheets.items():
    if {"index_spot", "halo"}.issubset(df.columns):
        counts[cible] = len(df)

print(f"Nombre d'onglets (souches en tapis) valides trouvés : {len(counts)}\n")
for cible, n in counts.items():
    print(f"{cible}: {n}")
print("\nTotal de lignes valides (toutes feuilles confondues):", sum(counts.values()))
print("\n-----------------------------------------------\n")

# TRAITEMENT ET FUSION
all_rows = []

for sheet in sheet_names:
    df = sheets[sheet]
    if not {"index_spot", "halo"}.issubset(df.columns):
        print(f"[IGNORÉ] Feuille '{sheet}' : structure incorrecte.")
        continue

    cible = normalize_cible_name(sheet)

    # Mapping des index spots
    df = df.copy()
    df["productrice"] = df["index_spot"].map(MAPPING)

    # Retire les lignes sans correspondance
    df = df[~df["productrice"].isna()].copy()

    # Conversion des halos
    df["halo"] = df["halo"].apply(halo_to_10_or_blank)

    # Assemblage du sous-tableau
    out = pd.DataFrame({
        "cible": cible,
        "productrice": df["productrice"].astype(str),
        "couple": df["productrice"].astype(str) + "_" + cible,
        "halo": df["halo"]
    })

    all_rows.append(out)

# CONCATÉNATION (sans export)
if all_rows:
    result = pd.concat(all_rows, ignore_index=True)
else:
    result = pd.DataFrame(columns=["cible", "productrice", "couple", "halo"])

# APERÇU FINAL
print("Aperçu des 20 premières lignes :\n")
print(result.head(20))
print(f"\nNombre total de couples traités : {len(result)}")

Nombre d'onglets (souches en tapis) valides trouvés : 38

1001: 40
1003: 40
1006: 40
1009: 40
1011: 40
1012: 40
1014 (2): 40
1015: 40
1017: 40
1019: 40
1020: 40
1025: 40
1028: 40
1032 (984 et 990 inversés): 40
1033: 40
1036: 40
1041: 40
1044: 40
1048: 40
1054: 40
952: 40
955: 40
958: 40
959: 40
982: 40
984: 40
985: 40
986: 40
987: 40
988: 40
990: 40
992: 40
993: 28
994: 20
995: 40
996: 40
998: 40
999: 40

Total de lignes valides (toutes feuilles confondues): 1488

-----------------------------------------------

Aperçu des 20 premières lignes :

   cible productrice     couple halo
0   1001         982   982_1001    0
1   1001         959   959_1001    0
2   1001         958   958_1001    0
3   1001         955   955_1001    0
4   1001         952   952_1001    0
5   1001         988   988_1001    0
6   1001         987   987_1001    0
7   1001         986   986_1001    0
8   1001         985   985_1001    0
9   1001         984   984_1001    0
10  1001         995   995_1001    0
11  

In [3]:
# On supprime les lignes contenant "M63" (qui n'est pas une souche)
# dans n'importe quelle colonne
df_filtre = result[~result.apply(lambda row: row.astype(str).str.contains("M63", na=False)).any(axis=1)]

print(f"Lignes initiales : {len(result)}")
print(f"Lignes après filtrage : {len(df_filtre)}")

Lignes initiales : 1488
Lignes après filtrage : 1452


In [4]:
Data = pd.read_excel("../data/Donnees_Reference/Tableau_complet_tests_croisés_Labo_MNHN.xlsx")
Data.rename(columns={"Unnamed: 0": ""}, inplace=True)
Data.head()

,,952,955,958,959,982,984,985,986,987,...,1020,1025,1028,1032,1033,1036,1041,1044,1048,1054
0,952,,,,,,X,,,,...,X,,O,O,X,X,X,X,,
1,955,O,,O,,X,,,,O,...,,,,,,,,,,
2,958,,,,,,,,,,...,,,,,,,,,,
3,959,O,,,,,,,,,...,,,,,,,,,,
4,982,,,,,,,,,,...,,,,,,,,,,


In [5]:
# Création du fichier Excel final avec deux feuilles
with pd.ExcelWriter("../data/sortie/variable_halo_VF.xlsx", engine="openpyxl") as writer:
    df_filtre.to_excel(writer, sheet_name="halo_image", index=False)
    Data.to_excel(writer, sheet_name="verif", index=False)

print("Fichier 'variable_halo_VF.xlsx' créé avec les feuilles 'halo_image' et 'verif'")

Fichier 'variable_halo_VF.xlsx' créé avec les feuilles 'halo_image' et 'verif'


Différences d’inhibition : SAM2 vs MNHN

In [6]:
# --------- UTILS ----------
def to_int64_or_na(series):
    """Force 0/1/NA en Int64 (entiers + NA)."""
    s = pd.to_numeric(series, errors="coerce")
    return s.astype("Int64")

def verif_cell_to_binary(val):
    """Interprétation de la cellule dans 'verif' : non vide → 1, vide → 0."""
    if pd.isna(val):
        return 0
    s = str(val).strip()
    return 1 if len(s) > 0 else 0

# --------- PRÉPARATION DES DONNÉES ----------
# halo_image (df_filtre)
df_halo = df_filtre.copy()

# Harmoniser types
df_halo["cible"] = df_halo["cible"].astype(str).str.strip()
df_halo["productrice"] = df_halo["productrice"].astype(str).str.strip()
df_halo["couple"] = df_halo["productrice"] + "_" + df_halo["cible"]

# halo en entiers 0/1/NA
df_halo["halo"] = to_int64_or_na(df_halo["halo"])

# verif (Data)
df_verif_raw = Data.copy()
first_col = df_verif_raw.columns[0]

# Matrice : index=cibles, colonnes=productrices
verif = df_verif_raw.set_index(first_col)
verif.index = verif.index.astype(str).str.strip()
verif.columns = verif.columns.astype(str).str.strip()

# --------- ALIGNEMENT & CORRECTION ----------
def expected_from_verif(row):
    cible = row["cible"]
    prod  = row["productrice"]
    try:
        val = verif.loc[cible, prod]
    except KeyError:
        val = pd.NA
    return verif_cell_to_binary(val)

df = df_halo.copy()
df["expected"] = df.apply(expected_from_verif, axis=1).astype("Int64")

# Compter & corriger
mod_counts = {"0->1":0, "1->0":0, "NA->1":0, "NA->0":0}

def correct_halo(row):
    h = row["halo"]
    e = row["expected"]

    if pd.isna(h):
        if e == 1:
            mod_counts["NA->1"] += 1
        else:
            mod_counts["NA->0"] += 1
        return e

    if h != e:
        if h == 0 and e == 1:
            mod_counts["0->1"] += 1
        elif h == 1 and e == 0:
            mod_counts["1->0"] += 1
        return e

    return h

df["halo_corrige"] = df.apply(correct_halo, axis=1).astype("Int64")

# --------- RAPPORT ----------
total_modifs = sum(mod_counts.values())
print("\n--- Vérification & corrections ---")
print(f"Total de lignes : {len(df)}")
print(f"Total de modifications : {total_modifs}")
for k, v in mod_counts.items():
    print(f"{k} : {v}")

if total_modifs > 0:
    print("\nExemples de corrections (jusqu'à 10):")
    examples = df.loc[df["halo"] != df["halo_corrige"], ["cible","productrice","couple","halo","expected","halo_corrige"]].head(10)
    print(examples.to_string(index=False))


--- Vérification & corrections ---
Total de lignes : 1452
Total de modifications : 436
0->1 : 363
1->0 : 2
NA->1 : 10
NA->0 : 61

Exemples de corrections (jusqu'à 10):
cible productrice   couple  halo  expected  halo_corrige
 1001         952 952_1001     0         1             1
 1001         990 990_1001     0         1             1
 1003         982 982_1003     0         1             1
 1003         958 958_1003     0         1             1
 1003         952 952_1003     0         1             1
 1003         987 987_1003     0         1             1
 1003         994 994_1003     0         1             1
 1003         990 990_1003     0         1             1
 1006         982 982_1006     0         1             1
 1006         959 959_1006     0         1             1


In [7]:
# Couples corrigés uniquement pour 1->0 ou NA->1
corrections = df.loc[
    ((df["halo"] == 1) & (df["halo_corrige"] == 0)) |   # 1 -> 0
    (df["halo"].isna() & (df["halo_corrige"] == 1))     # NA -> 1
]

# On garde seulement les infos utiles
corrections = corrections[["cible", "productrice", "couple", "halo", "halo_corrige"]]

print("Corrections 1->0 et NA->1 :")
print(corrections.to_string(index=False))

Corrections 1->0 et NA->1 :
cible productrice    couple  halo  halo_corrige
 1009         985  985_1009  <NA>             1
 1012        1020 1020_1012  <NA>             1
 1012        1036 1036_1012  <NA>             1
 1015         952  952_1015  <NA>             1
 1025         952  952_1025  <NA>             1
 1044         992  992_1044     1             0
 1044        1028 1028_1044     1             0
 1048         952  952_1048  <NA>             1
  985         952   952_985  <NA>             1
  993         994   994_993  <NA>             1
  994         955   955_994  <NA>             1
  995         952   952_995  <NA>             1
